# SnapChef - YOLOv8 Edge Vision Training
This notebook trains a lightweight YOLOv8 Nano model on a food ingredients dataset and exports it to `.tflite` for use in React Native (`react-native-fast-tflite`).

**Changed from the first version of this notebook:**
- Exports a **plain float32 `.tflite`** instead of `int8`-quantized. The int8 export used a newer Ultralytics/TFLite conversion path that produces ops only a very recent TFLite engine can run — `react-native-fast-tflite`'s iOS build is pinned to an older TensorFlowLiteC (2.17.0) that can't execute it (confirmed: the int8 model runs fine in Google's own newest TFLite runtime, but throws a generic `Invoke() failed` error in the app on both Simulator and a real device). Plain float32 avoids that newer quantization pattern entirely and is the most universally-compatible export TFLite has.
- **Backs up to Google Drive at two points** (right after training, and right after export) — last time, everything lived only on the Colab VM's temporary disk, which gets wiped when the runtime disconnects, and the trained weights + class list were lost with no way to recover them except reverse-engineering the labels from the dataset's own hosting service. This time nothing is lost if the runtime disconnects mid-notebook.
- **Sanity-checks the exported `.tflite` inside Colab** before you ever download it — runs real inference on a test image and shows the detected boxes, so a broken export is caught here instead of after a native rebuild on your end.
- Downloads `data.yaml` (the class list) alongside the model, so the labels never need to be recovered after the fact again.

## 1. Setup Environment

In [ ]:
!pip install -q ultralytics roboflow


## 2. Mount Google Drive (backup target)
Do this **first**, before anything else — everything downloaded/trained below gets backed up here as we go, so a disconnected runtime never means starting over from zero again.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
BACKUP_DIR = '/content/drive/MyDrive/SnapChefVision'
os.makedirs(BACKUP_DIR, exist_ok=True)
print(f"Backups will be saved to: {BACKUP_DIR}")


## 3. Download Dataset from Roboflow
1. Go to Roboflow Universe and find a food ingredients dataset (YOLOv8 format).
2. Click 'Download Dataset' -> 'Show Download Code'.
3. Paste your Roboflow API key and workspace/project details below.

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="mHMwqsFzwVtHmAeu2bTg")
project = rf.workspace("food-recipe-ingredient-images-0gnku").project("food-ingredients-dataset")
version = project.version(4)
dataset = version.download("yolov8")

# Back up data.yaml (the class list) immediately — this is the file that was
# lost entirely last time, forcing the labels to be reverse-engineered from
# Roboflow's API after the fact.
import shutil
shutil.copy(f"{dataset.location}/data.yaml", f"{BACKUP_DIR}/data.yaml")
print(f"Backed up data.yaml to {BACKUP_DIR}/data.yaml")


## 4. Train YOLOv8 Nano

In [ ]:
from ultralytics import YOLO

# Load the lightweight nano model
model = YOLO('yolov8n.pt')

# Train the model. We use imgsz=320 for faster mobile inference.
# Note: dataset.location contains the path to the downloaded data.yaml
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=50,
    imgsz=320,
    batch=16,
    project="snapchef_vision",
    name="edge_model"
)


In [ ]:
# Robust path lookup — different Ultralytics versions nest run folders
# differently (e.g. runs/detect/<project>/<name>/ vs <project>/<name>/
# directly), so find things by glob instead of assuming one fixed layout.
import glob
from pathlib import Path

def find_latest(pattern):
    matches = glob.glob(f"/content/**/{pattern}", recursive=True)
    assert matches, f"Could not find anything matching {pattern}"
    return Path(max(matches, key=lambda p: Path(p).stat().st_mtime))


## 5. Back Up Trained Weights to Drive
Do this immediately after training finishes, before anything else can go wrong — `best.pt` is the actual trained model; everything after this point (export, download) can be re-derived from it without retraining.

In [ ]:
import shutil

weights_src = find_latest("edge_model/weights/best.pt")
weights_backup = f"{BACKUP_DIR}/best.pt"
shutil.copy(weights_src, weights_backup)
print(f"Found trained weights at: {weights_src}")
print(f"Backed up trained weights to {weights_backup}")


## 6. Verify & Test the Model
Let's run the trained model on a test image to make sure it's working perfectly.

In [ ]:
from IPython.display import Image, display

# Grab a random image from the test set
test_images = glob.glob(f"{dataset.location}/test/images/*.jpg")
if len(test_images) > 0:
    test_image = test_images[0]

    # Run inference
    res = model.predict(source=test_image, imgsz=320, save=True, project="snapchef_vision", name="test_inference")

    # Display the result — use the actual save_dir Ultralytics reports back,
    # rather than assuming where it saved.
    display(Image(filename=str(Path(res[0].save_dir) / Path(test_image).name)))
else:
    print("No test images found.")


## 7. Export to TFLite (float32)
Plain float32 — **not** `int8` this time. Deliberately skipping quantization: it's the export path that produced ops `react-native-fast-tflite`'s pinned TensorFlowLiteC (2.17.0) can't execute. Float32 is larger (tens of MB instead of a few MB for a nano model) but that's a non-issue for a phone app, and it's the most universally-supported export TFLite has.

In [ ]:
# Export to TFLite — plain float32, no int8/half quantization
model.export(
    format='tflite',
    imgsz=320,
    int8=False,
    half=False,
)

tflite_path = find_latest("best_saved_model/*.tflite")
print(f"Exported model: {tflite_path}")


## 8. Sanity-Check the Exported .tflite
Run real inference through the exported TFLite file itself (not the original PyTorch model) on a test image, right here in Colab. If this produces sane-looking detections, the export is good and the file is safe to bring into the app.

In [ ]:
tflite_model = YOLO(tflite_path)
res = tflite_model.predict(source=test_image, imgsz=320, save=True, project="snapchef_vision", name="tflite_test_inference")
display(Image(filename=str(Path(res[0].save_dir) / Path(test_image).name)))

print("\nDetected classes in this test image:")
for r in res:
    for c in r.boxes.cls:
        print(" -", model.names[int(c)])


## 9. Back Up the Exported Model to Drive
Second backup point — in case the download step below fails or gets interrupted, the exported model (and its data.yaml) are already safe.

In [ ]:
import shutil

shutil.copy(tflite_path, f"{BACKUP_DIR}/best_float32.tflite")
print(f"Backed up exported model to {BACKUP_DIR}/best_float32.tflite")


## 10. Download the Model
Downloads both the `.tflite` model and its `data.yaml` (class list) directly to your computer. Put `best_float32.tflite` in `mobile/assets/models/` (replacing `best_int8.tflite`) and `data.yaml` alongside it for reference.

In [ ]:
from google.colab import files

files.download(tflite_path)
files.download(f"{dataset.location}/data.yaml")
